# Importation des données
On utilise un chemin relatif pour que tout le monde puisse reprendre exactement ce code simplement copiant le dépo github

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("..") / "data" / "raw" / "churn_modelling.csv"
df = pd.read_csv(DATA_PATH, sep = ";")
df.head(5)

# Importation des packages pour l'analyse

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import chi2_contingency

# Qualité des données 
Vérification de la structure globale des données : 

- Types des données
- Détection des valeurs manquantes 
- Détection des valeurs aberantes 
- Détection des doublons 
- Vérification de la logique des modalités pour les variables catégorielles

In [ ]:
df.info()

Aucune surprise à signaler au niveau du typage des variables.

In [ ]:
df.isna().sum()

Aucune valeurs manquantes dans le dataset.

In [ ]:
df.drop(columns="RowNumber").duplicated().sum()
df["CustomerId"].duplicated().sum()

Nous n'avons pas non plus de doublons. 

In [ ]:
print("Modas de geography : ",df["Geography"].unique())
print("Modas de gender = ", df["Gender"].unique())

Nos modalités catégorielles sont bien cohérentes avec la variable initiale. 
Enfin nous controlerons la présence de valeurs aberantes au travers des statistiques descriptives ci-dessous :

# Stats desc
Analyses descriptives préalable à tout travail de modélisation. On va notamment ici regarder :

1) La distribution des variables 
2) Les relations "basiques" existantes entre elles
3) La structure globale du jeu de données
4) La présence ou non de valeurs aberantes 

Cela afin de travailler proprement au moment du feature engineering et d'orienter la modélisation vers la solution la plus adaptée aux données. 

In [ ]:
# On créer 3 sous-df en fonction du type de variable pour faciliter l'EDA et travailler proprement.
vars_quanti = df.drop(["RowNumber", "CustomerId", "Surname", "Geography", "Gender", "HasCrCard","IsActiveMember"], axis = 1)
vars_quali = df[["Geography", "Gender"]]
vars_bin = df[["Exited", "HasCrCard","IsActiveMember"]]
vars_quanti.head(5)

Visualisons une de nos problématiques (très fréquente sur de la modélisation de churn) : déséquilibre au niveau de la variable à expliquer.

In [ ]:
(df["Exited"].value_counts(normalize=True) * 100).plot(
    kind="bar",
    figsize=(7, 5))

plt.title("Répartition des clients (en %)")
plt.xlabel("Exited")
plt.ylabel("Pourcentage (%)")
plt.xticks(rotation=0)
plt.show()

Nb : Nous savons que 80/20 sur de la modélisation de churn reste encore un cas d'école. 
En modélisant de vrais churn sur d'autres projets ou en entreprise notamment, nous avons traité des cas "réels" avec 2/3% de churn. Le but du projet étant ici de pratiquer le MLOps plus que de mettre en oeuvre nos compétences en modélisation, travailler sur un cas d'école ne pose pas de problèmes

## Analyse indépendante des quanti

On commence par analyser de manière indépendante les variables quantitatives : distributions, structures, liens entre elles et avec la variable cible

In [ ]:
vars_quanti.describe() 
# Nb : Au dela des statistiques descriptives pures, on voit que les variables expriment des unités et des grandeurs différentes : une standardisation sera 
# indispensable au moment de la modélisation pour ces dernières. 

Pas de valeurs aberrantes ici : 
- l'âge est compris entre 18 (minimum pour avoir un compte bancaire en Europe) et 92 ans ce qui est crédible.
- L'ancienneté au niveau de la fidelité semble cohérente
- Bien que le solde bancaire maximale puisse sembler élevé, on ne peut pas considérer ça comme aberrant un tel montant reste tout à fait possible
- Même chose pour le salaire : un minimum à 11 peut sembler étrange mais rien d'aberrant puisqu'on parle ici de salaire et non de revenus. Par exemple un chômeur peut très bien percevoir un revenu mais avoir un salaire quasi inexistant comme c'est le cas ici. 

In [ ]:
corr_matrix = vars_quanti.corr(method='pearson')

plt.figure(figsize=(10, 7))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='coolwarm',
    center=0,
    fmt='.2f')
plt.title("Matrice de corrélation")
plt.show()

Les liens statistiques "primaires" les plus forts avec la existants variable cible concernent l'âge et le solde disponible sur le compte en banque du client. 
Cependant les relations linéaires existantes restent relativement faible, si on choisit d'utiliser un modèle linéaire il sera donc surement pertinent d'ajouter des termes d'interactions. 

# Analyse indépendante des variables binaires 
Objectif : observer la liaison existante dans le dataset entre nos variables binaires

In [ ]:
vars_bin.describe()

In [ ]:
vars_quali.describe()

In [ ]:
vars_quali.groupby('Geography').describe()

In [ ]:
for col in vars_bin:
    if col != 'Exited':
        tableau = pd.crosstab(df[col], df['Exited'])

        chi2, p_value, dof, expected = chi2_contingency(tableau)
        n = tableau.sum().sum()
        min_dim = min(tableau.shape) - 1

        v_cramer = np.sqrt(chi2 / (n * min_dim))

        print(f"{col} : V = {v_cramer:.3f} | p-value = {p_value:.5f}")

Association respectivement inexistante et faible entre le churn et ces deux variables ce qui semble logique d'un pdv purement logique au vu de ce que ces variables représentent (également compte tenu du fait que la majorité des clients, en churn comme fidèles, ont une carte de crédit).  

In [ ]:
# On peut également "dézoomer" et regarder les liens existants entre l'ensemble des variables catégorielles

vars_cat = [
    'Gender',
    'Geography',
    'Exited',
    'HasCrCard',
    'IsActiveMember']

def v_de_cramer(x, y):
    tableau = pd.crosstab(x, y)
    chi2 = chi2_contingency(tableau)[0]
    n = tableau.to_numpy().sum()
    min_dim = min(tableau.shape) - 1
    return np.sqrt(chi2 / (n * min_dim))


matrice = pd.DataFrame(
    index=vars_cat,
    columns=vars_cat,
    dtype=float)

for col1 in vars_cat:
    for col2 in vars_cat:
        matrice.loc[col1, col2] = v_de_cramer(
            df[col1], df[col2])


plt.figure(figsize=(9, 7))
sns.heatmap(
    matrice,
    annot=True,
    cmap='coolwarm',
    vmin=0,
    vmax=1,
    fmt='.2f'
)
plt.title("Matrice d'association des V de Cramer")
plt.show()

Au global on voit que de faibles associations existent entre la variable cible et des variables catégorielles. 
Si nous choisissons de faire un modèle linéaire il faudra surement ajouter des termes d'interactions

In [ ]:
# Comme il s'agit du lien catégorielle le plus fort, ça p-e intéressant de regarder plus en détail l'association entre Exited et Geography :

geography_dummies = pd.get_dummies(
    df['Geography'],
    prefix='Geography')

# on va donc faire le calcul du V de Cramér pour chaque pays
for col in geography_dummies.columns:
    tableau = pd.crosstab(
        geography_dummies[col],
        df['Exited'])
    
    chi2 = chi2_contingency(tableau)[0]
    n = tableau.to_numpy().sum()
    min_dim = min(tableau.shape) - 1
    v_cramer = np.sqrt(chi2 / (n * min_dim))

    print(f"{col} : V de Cramer = {v_cramer:.3f}")

In [ ]:
df.groupby('Geography')['Exited'].agg(
    taux_churn='mean',
    nombre_clients='count'
).assign(
    taux_churn=lambda x: x['taux_churn'] * 100)

Effectivement le taux de churn est largement supérieur chez les clients allemands (un client sur 3). 
Il faudra veiller à ce que ce lien important soit correctement modelisé

# Test de quelques recodages + visualisations 

Si un lien linéaire existe déjà entre le churn et l'âge des clients, ça peut être intéressant de voir comment ce dernier se découpe en fonction de grandes catégories d'âge. C'est ce qu'on va voir ici en recodant l'âge par tranche puis en analysant sont lien avec le churn sous ce prisme.

In [ ]:
bins = list(range(18, 101, 10))

df["Age_classe"] = pd.cut(
    df["Age"],
    bins=bins,
    right=False,
    include_lowest=True) 

Une fois qu'on a decoupé l'âge en 10 tranches on peut visualiser sa distribution : 

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    data=df,
    x="Age",
    bins=bins,
    edgecolor="black"
)

plt.title("Distribution des clients par classe d'âge")
plt.xlabel("Âge (classes de 10 ans)")
plt.ylabel("Nombre de clients")
plt.xticks(bins)
plt.grid(axis="y", alpha=0.3)
plt.show()

La distribution étant maintenant connue, nous pouvons regarder le taux de churn par tranche pour voir si un pattern est reconnaissable ici : 

In [ ]:
churn_age = df.groupby("Age_classe", observed=False)["Exited"].mean() * 100
print(churn_age)

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    x=churn_age.index.astype(str),
    y=churn_age.values
)

plt.title("Taux de churn par tranche d'âge")
plt.xlabel("Tranche d'âge")
plt.ylabel("Taux de churn (%)")
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)

plt.show()

On observe effectivement que le taux de churn à une distribution particulière en fonction de la tranche d'âge à laquelle appartient le client. 
Il faudra tenir compte de ce phénomène au moment de la modélisation. Ici un recodage de la variable âge en x modalités catégorielles (regroupant les tranches d'ages avec une moyenne de churn non significativement différentes les unes des autres) peut-être intéressant 

- Procédons au même type d'analyse pour la variable EstimatedSalary :

In [ ]:
pas = 25000

bins = np.arange(
    0,
    (df["EstimatedSalary"].max() // pas + 2) * pas,
    pas
)
df["Salaire_classe"] = pd.cut(
    df["EstimatedSalary"],
    bins=bins,
    right=False,
    include_lowest=True
)

Grâce à ce découpage (pour l'instant arbitraire), nous pouvons calculer le taux de churn par tranche et visualiser ce dernier : 

In [ ]:
churn_revenus = df.groupby(
    "Salaire_classe", observed=False
).agg(
    nombre_clients=("Exited", "count"),
    clients_partis=("Exited", "sum"),
    taux_churn=("Exited", "mean")
)

churn_revenus["taux_churn"] *= 100

churn_revenus

On voit que ici le taux de churn reste indépendant des tranches de revenus même si il augmente très légèrement avec ce dernier. La visualisation est donc inutile tout comme le découpage en classe de cette variable.

- Même analyse pour la variable Balance 
(toujours via un découpage arbitraire, si on voit que ce traitement est intéressant on cherchera p-e à l'optimiser après)

In [ ]:
pas = 25000

bins = np.arange(
    0,
    (df["Balance"].max() // pas + 2) * pas,
    pas
)
df["Balance_classe"] = pd.cut(
    df["Balance"],
    bins=bins,
    right=False,
    include_lowest=True
)

In [ ]:
churn_balance = df.groupby(
    "Balance_classe", observed=False
).agg(
    nombre_clients=("Exited", "count"),
    clients_partis=("Exited", "sum"),
    taux_churn=("Exited", "mean")
)

churn_balance["taux_churn"] *= 100

churn_balance

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    x=churn_balance.index.astype(str),
    y=churn_balance["taux_churn"]
)

plt.title("Taux de churn selon le montant sur le compte")
plt.xlabel("Tranches de solde")
plt.ylabel("Taux de churn (%)")
plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

On constate ici que passé un certain seuil (200 000), le montant du solde sur le compte bancaire s'accompagne d'une explosion du taux de churn. 
Bien que cela représente relativement peu d'individus (cf stats desc) on va p-e devoir tenir compte de ce phénomène pour améliorer la finesse du modèle.
Le vrai phénomène ici est la tranche 0-25 0000 qui regroupe 3000 clients et qui à un taux de churn significativement inféiruer (13%) à l'enesemble. La modélisation devra en tenir compte. Comme cela représente beaucoup de cas, il p-e inéterssant de créer une variable indicatrice

- Enfin menons une analyse du même type sur le score de crédit : 

In [ ]:
pas = 50

bins = np.arange(
    350,
    (df["CreditScore"].max() // pas + 2) * pas,
    pas
)
df["Score_classe"] = pd.cut(
    df["CreditScore"],
    bins=bins,
    right=False,
    include_lowest=True
)

In [ ]:
churn_score = df.groupby(
    "Score_classe", observed=False
).agg(
    nombre_clients=("Exited", "count"),
    clients_partis=("Exited", "sum"),
    taux_churn=("Exited", "mean")
)

churn_score["taux_churn"] *= 100

churn_score

Fait marquant : le taux de churn est de 100% pour les clients très risqués (score de crédit les plus faibles). Il faudra vérifier que , idéalement le modèle capte bien cette nuance (sinon possible d'ajouter une variable pour le modéliser mais concerne très peu de client donc à voir).

In [ ]:
df_mauvais = df[df["CreditScore"]<410]
df_mauvais.head(10)

# Taux de churn par modalités pour les variables discrètes 
Calculons également, à l'image de ce que nous venons de faire pour les variables catégorielles (+ tenure qui est un entre deux) le taux de churn par modalités

In [ ]:
variables = [
    "NumOfProducts",
    "Gender",
    "IsActiveMember"]

for var in variables:
    print(f"Taux de churn selon {var}")

    resultat = df.groupby(var).agg(
        nombre_clients=("Exited", "count"),
        clients_partis=("Exited", "sum"),
        taux_churn=("Exited", "mean")
    )

    resultat["taux_churn"] *= 100

    display(resultat.round(2))

Faits marquants : le taux de churn explose pour les détenteurs de 3 e 4 produits, est à l'inverse bien dessous de la moyenne pour ceux qui en détiennent 2 + les Hommes smblent être moins propices au phénomène. 

# Marche à suivre pour la suite : 

1) Exclure les variables RowNumber, CustomerId et Surname qui n'ont aucun intêrets pour la modélisation. 
2) Standardiser les variables quantitatives pour tester les modèles linéaires/à base de distance 
3) Garder notre découpage de l'âge en 10 tranches et faire nos premiers tests. Si besoin réaliser un découpage + fin et intelligent
4) Vérifier que les modèles qu'on va tester et qui semblent pertinent modélisent bien A) l'explosion du churn à partir de 200 000 de solde bancaire mais surtout la sous représentation du churn chez les soldes entre 0 et 25 000 B) l'explosion du churn à 4 produits et au contraire sa forte contraction à 2 produits C) la sous représentation significative des Hommes dans ce phénomène D) l'explosion du churn chez les clients les plus à risque. Si ce n'est pas le cas, voir pour ajouter des variables permettant de capturer ces phénomènes (en faisant attention au sur-apprentissage)
5) Bien faire tout les découpages et recodage sur le jeu de train pour éviter tout problèmes. Cad split stratifié train/test avec une graine fixe, avant tout calcul appris (standardisation, regroupements)